# pydantic_graph smoke test

`pydantic_graph` is the type-hint-based graph library powering the Pydantic AI agent loop. Graphs are constructed with `GraphBuilder` from typed step functions (and optionally `BaseNode` subclasses), then executed via `Graph` / `GraphRun`.

In [ ]:
# Example 1: BaseNode subclass identity + End result wrapper.
from pydantic import BaseModel
from pydantic_graph import BaseNode, End, GraphRunContext


class State(BaseModel):
    value: int = 0


class ProcessNode(BaseNode[State, None, State]):
    async def run(self, ctx: GraphRunContext[State, None]) -> End[State]:
        ctx.state.value = 42
        return End(data=ctx.state)


node = ProcessNode()
print(f"node id: {node.get_node_id()}")
assert node.get_node_id() == 'ProcessNode'

end = End(data=State(value=100))
print(f"End.data.value = {end.data.value}")
assert end.data.value == 100

In [ ]:
# Example 2: multiple BaseNode subclasses carry distinct node ids.
from pydantic import BaseModel
from pydantic_graph import BaseNode, End, GraphRunContext


class CountState(BaseModel):
    count: int = 0


class IncrementNode(BaseNode[CountState, None, CountState]):
    async def run(self, ctx: GraphRunContext[CountState, None]) -> End[CountState]:
        ctx.state.count += 1
        return End(data=ctx.state)


class ResetNode(BaseNode[CountState, None, CountState]):
    async def run(self, ctx: GraphRunContext[CountState, None]) -> End[CountState]:
        ctx.state.count = 0
        return End(data=ctx.state)


ids = {IncrementNode().get_node_id(), ResetNode().get_node_id()}
print(f"node ids: {sorted(ids)}")
assert ids == {'IncrementNode', 'ResetNode'}

In [ ]:
# Example 3: build + run a single-step graph with GraphBuilder (the primary API).
# In Jupyter's already-running event loop, use top-level `await graph.run(...)`
# (run_sync would raise "event loop is already running").
from pydantic_graph import GraphBuilder, StepContext

builder = GraphBuilder(state_type=None, input_type=int, output_type=int)


@builder.step
async def double(ctx: StepContext[None, None, int]) -> int:
    return ctx.inputs * 2


builder.add_edge(builder.start_node, double)
builder.add_edge(double, builder.end_node)
graph = builder.build()

result = await graph.run(inputs=21)
print(f"await graph.run(inputs=21) -> {result}")
assert result == 42

In [ ]:
# Example 4: chain two steps + render the graph to mermaid.
from pydantic_graph import GraphBuilder, StepContext

builder = GraphBuilder(state_type=None, input_type=int, output_type=str)


@builder.step
async def add_one(ctx: StepContext[None, None, int]) -> int:
    return ctx.inputs + 1


@builder.step
async def stringify(ctx: StepContext[None, None, int]) -> str:
    return f"value={ctx.inputs}"


builder.add_edge(builder.start_node, add_one)
builder.add_edge(add_one, stringify)
builder.add_edge(stringify, builder.end_node)
graph = builder.build()

assert await graph.run(inputs=41) == 'value=42'

diagram = graph.render()
print(diagram[:120])
assert isinstance(diagram, str) and 'add_one' in diagram and 'stringify' in diagram

In [ ]:
# Example 5: the graph error classes are exported and raise/catch cleanly.
from pydantic_graph import GraphRuntimeError, GraphSetupError

assert issubclass(GraphRuntimeError, Exception)
assert issubclass(GraphSetupError, Exception)

try:
    raise GraphRuntimeError('graph execution failed')
except GraphRuntimeError as e:
    print(f"caught GraphRuntimeError: {e}")
    assert str(e) == 'graph execution failed'

print("pydantic_graph smoke test PASS")